In [1]:
from datetime import datetime

from migration.schema_extractor import SchemaExtractor
from transformers.utils import replace_variables_in_strings, replace_variables_in_comments

s = '2026-03-18 15:57:19.062'
s = datetime.strptime(s, '%Y-%m-%d %H:%M:%S.%f').strftime('%Y-%m-%dT%H:%M:%S.%f')[:-3]
print(s)

2026-03-18T15:57:19.062


In [2]:
import yaml
from paths import VARIABLE_CONFIG_PATH

with open(VARIABLE_CONFIG_PATH, 'r', encoding='utf-8') as f:
    variable_mapping = yaml.safe_load(f)

In [3]:
import sqlglot


# 1. Đọc nội dung tệp SQL
with open(r"C:\Users\dungp\projects\hql_spark_bridge\docs\reference\logic_cleaner\main.sql", 'r') as file:
    sql_content = file.read()

original_ast = sqlglot.parse(sql_content, read='hive')

In [4]:
from transformers.utils import replace_table_identifier, strip_partition_clauses

# from src.transformers.utils import replace_table_identifier

ast_bien_doi = replace_table_identifier(
    node=original_ast[0],
    old_schema="${com_schema}",
    old_table="t_mhbos_m_client",
    new_schema="${com_schema}",
    new_table="temp_t_mhbos_m_client_consolidated",
    dialect="hive" # Hoặc spark
)

ast_bien_doi = strip_partition_clauses(ast_bien_doi)

# In ra kết quả
print(ast_bien_doi.sql(dialect="hive", pretty=True))

/* 3.2 insert data to target table */
WITH today_accounts AS (
  SELECT DISTINCT
    client_no
  FROM ${com_schema}.temp_t_mhbos_m_client
)
INSERT INTO ${com_schema}.temp_t_mhbos_m_client_consolidated
SELECT
  client_no,
  clean_rule_flag,
  primary_identification_type,
  primary_identification_no,
  secondary_identification_type,
  secondary_identification_no,
  customer_name,
  customer_name_concatenate,
  client_name,
  client_name1,
  client_name2,
  client_name3,
  mobile_no,
  fax_no,
  tel_no_home,
  tel_no_office,
  date_of_birth,
  race,
  email_1,
  email_2,
  email_3,
  email_4,
  email_5,
  email_6,
  email_7,
  email_8,
  email_9,
  email_10,
  sex,
  addr1,
  addr2,
  addr3,
  addr4,
  postcode,
  city,
  state,
  perm_addr1,
  perm_addr2,
  perm_addr3,
  perm_addr4,
  perm_postcode,
  perm_city,
  perm_state,
  noms_ind,
  cleaned_nominees_name,
  principal_name,
  intermediary_name,
  beneficiary_name,
  nominees_type,
  pledged_securities_flag,
  id_type,
  ic_no_new,


In [5]:
replace_variabled = replace_variables_in_comments(original_ast[0], variable_mapping)


with open(r"C:\Users\dungp\projects\hql_spark_bridge\docs\reference\logic_cleaner\function_test\hilarious_comment.sql", "w") as f:
    f.write(replace_variabled.sql(dialect='hive', pretty=True) + ";")

In [6]:
from migration.key_detector import KeyDetectorV2
from utils.file_utils import parse_file_name
from migration.decomposer import SqlDecomposer
from paths import *
from src.utils.source_rule_loader import load_all_source_rules

# input_file = PROJECT_ROOT / "docs" / "datalake_old" /"dml" / "com_r_k2_cif_alias.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "com" /  "com_t_m21_a_customer.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "com" /  "com_t_mhbos_m_client.sql"
input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_contact.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_customer.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_risk_profile.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_customer_employment.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_customer_employment.sql"
file_name = os.path.basename(input_file).replace('.sql', '')
layer, sub_layer, source_name, base_table = parse_file_name(input_file)
output_root = PROJECT_ROOT / "output" / "migration"

all_source_rules = load_all_source_rules()
source_rules = load_all_source_rules()[source_name] if source_name in all_source_rules else all_source_rules['default']

print(f"File gốc tại: {input_file}")

# def run_migration_pipeline():
# ==========================================
# BƯỚC 1: BÓC TÁCH SQL (DECOMPOSER)
# ==========================================
decomposer = SqlDecomposer(source_rules)
decomposed_script = decomposer.decompose(input_file)

# Ghi file sub-SQL ra ổ đĩa
KeyDetectorV2().detect(decomposed_script, source_rules)


File gốc tại: C:\Users\dungp\projects\datalake-script\dml\cur\cur_dim_contact.sql


{'logical_primary_key': ['OWNER_ID', 'CONTACT_OWNER_TYPE', 'CONTACT_TYPE'],
 'frequency': 2,
 'confidence': 'HIGH',
 'reasoning': 'Found ROW_NUMBER() PARTITION BY matching this exact group 2 times.'}

In [7]:
from collections import Counter

c= Counter(a=4, b=2, c=0, d=-2)

c.most_common(1)

[('a', 4)]

In [8]:
schema = KeyDetectorV2()._get_schema(decomposed_script, source_rules)
schema

['owner_id',
 'contact_owner_type',
 'contact_type',
 'contact_value',
 'contact_name',
 'contact_create_date',
 'contact_update_date',
 'line_of_business',
 'source_name',
 'source_record_id',
 'sequence_no',
 'etl_timestamp']

In [9]:
from src.migration.ddl_resolver import DdlResolver
from src.migration.schema_extractor import SchemaExtractor

ddl_resolver = DdlResolver(source_rules = source_rules)
ddl_path = ddl_resolver.resolve_ddl_path(input_file)

schema_extractor = SchemaExtractor(source_rules)
columns = schema_extractor.extract(ddl_path)

[column['name'] for column in columns]

['owner_id',
 'contact_owner_type',
 'contact_type',
 'contact_value',
 'contact_name',
 'contact_create_date',
 'contact_update_date',
 'line_of_business',
 'source_name',
 'source_record_id',
 'sequence_no',
 'etl_timestamp']